# Earthquake Explorer

Explore a frozen USGS M2.5+ weekly snapshot. No account, API key, or data download is needed once Binder starts. Choose **Run > Run All Cells** for a complete demonstration. Change the search values below and rerun that cell to explore.

For the full interactive menu, open **File > New > Terminal**, enter `python earthquake_explorer.py`, and follow the prompts.

This is a historical snapshot, not a live alert service. Place searches match text, not distance or political boundaries.

In [1]:
import json
from pathlib import Path
from earthquake_explorer import load_earthquakes, find_earthquakes, format_results
metadata = json.loads(Path('data/provenance.json').read_text())
events, skipped = load_earthquakes()
print('Snapshot retrieved:', metadata['retrieved_at_utc'])
print('Event times:', metadata['earliest_event_utc'], 'to', metadata['latest_event_utc'])
print('Loaded:', len(events), '| Skipped:', skipped)

Snapshot retrieved: 2026-11-09T18:58:39.300992+00:00
Event times: 2026-11-02T19:01:12.785Z to 2026-11-09T18:48:08.388Z
Loaded: 322 | Skipped: 1


## Search
Change LOCATION and MINIMUM_MAGNITUDE. An empty location matches all places. Magnitude must be a finite number from 2.5 to 10. Results are sorted strongest first; only the first ten are displayed.

In [2]:
LOCATION = 'Alaska'
MINIMUM_MAGNITUDE = 3.0
try:
    matches = find_earthquakes(events, LOCATION, MINIMUM_MAGNITUDE)
    print(format_results(matches))
except ValueError as error:
    print('Invalid search:', error)

Matching earthquakes: 38
Strongest magnitude: 4.70
Mean reported magnitude: 3.45
The mean describes reported magnitudes, not average energy.
Showing 10 of 38 matches:
UTC time | Magnitude | Depth (km) | Location
2026-11-06T02:35:25.631Z | 4.70 | 27.96 | 139 km SSE of False Pass, Alaska
2026-11-05T23:04:57.395Z | 4.60 | 16.13 | 147 km SSE of False Pass, Alaska
2026-11-04T12:22:50.079Z | 4.30 | 11.59 | 78 km S of Nikolski, Alaska
2026-11-04T19:22:20.560Z | 4.20 | 74.50 | 30 km S of Adak, Alaska
2026-11-07T19:45:08.410Z | 4.00 | 27.90 | 203 km SE of Unalaska, Alaska
2026-11-05T08:32:02.979Z | 3.90 | 40.64 | 68 km S of Unalaska, Alaska
2026-11-05T09:14:44.556Z | 3.80 | 28.74 | 116 km SW of Nikolski, Alaska
2026-11-08T05:17:38.948Z | 3.80 | 3.50 | 73 km NNW of Chalkyitsik, Alaska
2026-11-03T16:48:57.818Z | 3.60 | 35.00 | 67 km SE of Adak, Alaska
2026-11-05T08:59:29.792Z | 3.60 | 43.01 | 115 km SW of Nikolski, Alaska


## No-match example

In [3]:
print(format_results(find_earthquakes(events, 'Atlantis', 2.5)))

No earthquakes matched your search in this snapshot.


## Full commented program
The same file runs both locally and in the Binder terminal.

In [4]:
from IPython.display import Code, display
display(Code(filename='earthquake_explorer.py', language='python'))

"""Explore a bundled USGS earthquake snapshot without a network connection."""

import csv
import math
from pathlib import Path

DATA_FILE = Path(__file__).resolve().parent / 'data' / 'earthquakes.csv'


def load_earthquakes(path=DATA_FILE):
    """Read valid earthquake rows; count rows unsuitable for this explorer."""
    earthquakes = []
    skipped = 0
    with open(path, encoding='utf-8', newline='') as source:
        reader = csv.DictReader(source)
        required = {'id', 'time', 'place', 'mag', 'depth', 'type'}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError('The CSV is missing required columns.')
        seen_ids = set()
        for row in reader:
            # Non-earthquake events and incomplete records must not affect totals.
            try:
                magnitude = float(row['mag'])
                depth = float(row['depth'])
                event_id = row['id'].strip()
                place = row['place'].strip()
                event_time = row['time'].strip()
                valid = (row['type'] == 'earthquake' and event_id and place
                         and event_time and math.isfinite(magnitude)
                         and math.isfinite(depth) and event_id not in seen_ids)
            except (ValueError, TypeError, AttributeError):
                valid = False
            if not valid:
                skipped += 1
                continue
            seen_ids.add(event_id)
            earthquakes.append({'id': event_id, 'time': event_time,
                                'place': place, 'magnitude': magnitude,
                                'depth': depth})
    return earthquakes, skipped


def parse_minimum(text):
    """Accept only a finite magnitude between 2.5 and 10, inclusive."""
    try:
        number = float(text)
    except (ValueError, TypeError):
        raise ValueError('Enter a number from 2.5 to 10.') from None
    if not math.isfinite(number) or not 2.5 <= number <= 10:
        raise ValueError('Enter a finite number from 2.5 to 10.')
    return number


def find_earthquakes(earthquakes, location, minimum):
    """Match a location substring and include events at the magnitude boundary."""
    minimum = parse_minimum(minimum)
    location = location.strip().casefold()
    matches = [event for event in earthquakes
               if location in event['place'].casefold()
               and event['magnitude'] >= minimum]
    # Strongest first, with time and ID providing reproducible tie ordering.
    return sorted(matches, key=lambda event: (-event['magnitude'],
                                              event['time'], event['id']))


def format_results(matches, limit=10):
    """Summarize all matches while keeping the displayed event list manageable."""
    if not matches:
        return 'No earthquakes matched your search in this snapshot.'
    strongest = max(event['magnitude'] for event in matches)
    average = sum(event['magnitude'] for event in matches) / len(matches)
    lines = [f'Matching earthquakes: {len(matches)}',
             f'Strongest magnitude: {strongest:.2f}',
             f'Mean reported magnitude: {average:.2f}',
             'The mean describes reported magnitudes, not average energy.',
             f'Showing {min(limit, len(matches))} of {len(matches)} matches:',
             'UTC time | Magnitude | Depth (km) | Location']
    for event in matches[:limit]:
        lines.append(f"{event['time']} | {event['magnitude']:.2f} | "
                     f"{event['depth']:.2f} | {event['place']}")
    return '\n'.join(lines)


def main(input_fn=input, output_fn=print, data_path=DATA_FILE):
    """Run a repeating search menu, recovering from ordinary input mistakes."""
    try:
        earthquakes, skipped = load_earthquakes(data_path)
    except (OSError, UnicodeError, ValueError, csv.Error) as error:
        output_fn(f'Cannot load the earthquake snapshot: {error}')
        return 1
    output_fn('EARTHQUAKE EXPLORER')
    outp

## Offline checks
Tests use local fixtures and the bundled CSV. They require no network or credentials.

In [5]:
import subprocess, sys
check = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], capture_output=True, text=True)
print(check.stdout + check.stderr)
assert check.returncode == 0

test_bad_rows_and_duplicates_are_skipped (test_explorer.ExplorerTests.test_bad_rows_and_duplicates_are_skipped) ... ok
test_blank_location_and_high_threshold (test_explorer.ExplorerTests.test_blank_location_and_high_threshold) ... ok
test_empty_data (test_explorer.ExplorerTests.test_empty_data) ... ok
test_eof_exits_cleanly (test_explorer.ExplorerTests.test_eof_exits_cleanly) ... ok
test_inclusive_boundary_and_case_insensitive_location (test_explorer.ExplorerTests.test_inclusive_boundary_and_case_insensitive_location) ... ok
test_invalid_magnitudes (test_explorer.ExplorerTests.test_invalid_magnitudes) ... ok
test_menu_recovers_from_input_errors (test_explorer.ExplorerTests.test_menu_recovers_from_input_errors) ... ok
test_missing_columns_and_missing_file (test_explorer.ExplorerTests.test_missing_columns_and_missing_file) ... ok
test_no_results (test_explorer.ExplorerTests.test_no_results) ... ok
test_result_limit_does_not_change_summary (test_explorer.ExplorerTests.test_result_limit_do

Source: https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php

The data retrieval time, original URL, event range, and SHA-256 are recorded in `data/provenance.json`. The mean of reported magnitudes is a descriptive statistic, not mean earthquake energy. Binder sessions are temporary; download edits you want to retain.